# Proof of Concept: Generative Recommender и HSTU

Самостоятельная учебная реализация ключевых узлов статьи **Actions Speak Louder than Words**.
Мы не импортируем авторский репозиторий. Ноутбук строит синтетический event stream, решает ranking и retrieval одной causal моделью, сравнивает HSTU с Transformer и проверяет системные инварианты.

**Граница воспроизведения:** это dense PyTorch на малом каталоге, а не custom ragged kernels, H100 benchmark или trillion-parameter model. Числа ноутбука нельзя сопоставлять с production-таблицами статьи.



## 1. Окружение и воспроизводимость

Нужен `torch>=2.1`. На CPU полный запуск занимает порядка минуты или меньше. Все генераторы зафиксированы; алгоритмы PyTorch переведены в deterministic режим, где это доступно.



In [1]:
import copy
import math
import random
import time
from dataclasses import dataclass

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Установите PyTorch: python -m pip install torch") from exc

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch={torch.__version__}; device={DEVICE}; threads={torch.get_num_threads()}")



torch=2.13.0; device=cpu; threads=1


## 2. Синтетический поток и temporal validation

В каталоге 48 объектов из 6 тематик. У пользователя две предпочитаемые тематики; positive action повышает вероятность продолжить локальный item-cycle. Это создаёт одновременно:

- задачу ranking: предсказать `skip/engage` для уже показанного target item;
- задачу retrieval: после действия предсказать следующий положительный item.

Для каждого пользователя первые 14 событий дают train losses. Последние 4 — validation targets, но causal context включает предшествующую train-историю. Случайного interaction split нет.



In [2]:
NUM_USERS = 256
NUM_ITEMS = 48
NUM_CATEGORIES = 6
ITEMS_PER_CATEGORY = NUM_ITEMS // NUM_CATEGORIES
NUM_INTERACTIONS = 18
TRAIN_CUTOFF = 14


def generate_stream(num_users=NUM_USERS, seq_len=NUM_INTERACTIONS):
    generator = torch.Generator().manual_seed(SEED)
    items = torch.empty(num_users, seq_len, dtype=torch.long)
    actions = torch.empty(num_users, seq_len, dtype=torch.long)
    timestamps = torch.empty(num_users, seq_len, dtype=torch.float32)

    for user in range(num_users):
        preferred = torch.randperm(NUM_CATEGORIES, generator=generator)[:2].tolist()
        now = 0.0
        previous_item = None
        previous_action = 0
        for step in range(seq_len):
            if step == 0 or torch.rand((), generator=generator).item() > 0.72:
                if torch.rand((), generator=generator).item() < 0.82:
                    category = preferred[int(torch.randint(2, (), generator=generator))]
                else:
                    category = int(torch.randint(NUM_CATEGORIES, (), generator=generator))
                offset = int(torch.randint(ITEMS_PER_CATEGORY, (), generator=generator))
                item = category * ITEMS_PER_CATEGORY + offset
            elif previous_action == 1:
                category = previous_item // ITEMS_PER_CATEGORY
                local = (previous_item % ITEMS_PER_CATEGORY + 1) % ITEMS_PER_CATEGORY
                item = category * ITEMS_PER_CATEGORY + local
            else:
                category = preferred[int(torch.randint(2, (), generator=generator))]
                item = category * ITEMS_PER_CATEGORY + int(
                    torch.randint(ITEMS_PER_CATEGORY, (), generator=generator)
                )

            category = item // ITEMS_PER_CATEGORY
            engage_probability = 0.88 if category in preferred else 0.12
            action = int(torch.rand((), generator=generator).item() < engage_probability)
            now += float(torch.randint(1, 11, (), generator=generator))

            items[user, step] = item
            actions[user, step] = action
            timestamps[user, step] = now
            previous_item, previous_action = item, action

    return items, actions, timestamps


all_items, all_actions, all_times = generate_stream()
print("items/actions/times:", all_items.shape, all_actions.shape, all_times.shape)
print("positive rate:", round(all_actions.float().mean().item(), 3))
print("first user items:  ", all_items[0].tolist())
print("first user actions:", all_actions[0].tolist())



items/actions/times: torch.Size([256, 18]) torch.Size([256, 18]) torch.Size([256, 18])
positive rate: 0.849
first user items:   [4, 2, 26, 27, 28, 29, 30, 3, 6, 30, 31, 31, 24, 25, 26, 25, 26, 27]
first user actions: [0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1]


## 3. Специфичный preprocessing: interleaving item/action

Item tokens имеют ID `1..NUM_ITEMS`, action tokens — отдельные ID после каталога, `0` зарезервирован под PAD. Для события $i$ получаем позиции $[\Phi_i, a_i]$.

- `action_targets` определён только на позиции $\Phi_i$;
- `item_targets` определён на позиции $a_i$, если следующее действие положительно;
- `-100` исключает позицию из `cross_entropy`.



In [3]:
PAD_ID = 0
ITEM_OFFSET = 1
ACTION_OFFSET = ITEM_OFFSET + NUM_ITEMS
VOCAB_SIZE = ACTION_OFFSET + 2
IGNORE_INDEX = -100


@dataclass
class PackedBatch:
    token_ids: torch.Tensor
    type_ids: torch.Tensor
    timestamps: torch.Tensor
    padding_mask: torch.Tensor
    action_targets: torch.Tensor
    item_targets: torch.Tensor

    def to(self, device):
        return PackedBatch(**{name: value.to(device) for name, value in self.__dict__.items()})


def pack_interleaved(items, actions, timestamps, target_start=0):
    batch_size, steps = items.shape
    length = 2 * steps
    token_ids = torch.full((batch_size, length), PAD_ID, dtype=torch.long)
    type_ids = torch.zeros((batch_size, length), dtype=torch.long)
    event_times = torch.zeros((batch_size, length), dtype=torch.float32)
    action_targets = torch.full((batch_size, length), IGNORE_INDEX, dtype=torch.long)
    item_targets = torch.full((batch_size, length), IGNORE_INDEX, dtype=torch.long)

    token_ids[:, 0::2] = items + ITEM_OFFSET
    token_ids[:, 1::2] = actions + ACTION_OFFSET
    type_ids[:, 1::2] = 1
    event_times[:, 0::2] = timestamps
    event_times[:, 1::2] = timestamps
    action_targets[:, 0::2] = actions

    # Retrieval supervision: следующий item считаем релевантным только при positive action.
    for step in range(steps - 1):
        valid_next = actions[:, step + 1].bool()
        position = 2 * step + 1
        item_targets[valid_next, position] = items[valid_next, step + 1]

    if target_start > 0:
        boundary = 2 * target_start
        action_targets[:, :boundary] = IGNORE_INDEX
        item_targets[:, :boundary] = IGNORE_INDEX

    return PackedBatch(
        token_ids=token_ids,
        type_ids=type_ids,
        timestamps=event_times,
        padding_mask=token_ids.ne(PAD_ID),
        action_targets=action_targets,
        item_targets=item_targets,
    )


train_batch = pack_interleaved(
    all_items[:, :TRAIN_CUTOFF], all_actions[:, :TRAIN_CUTOFF], all_times[:, :TRAIN_CUTOFF]
)
valid_batch = pack_interleaved(all_items, all_actions, all_times, target_start=TRAIN_CUTOFF)

print("train tokens:", train_batch.token_ids.shape)
print("train action labels:", int(train_batch.action_targets.ne(IGNORE_INDEX).sum()))
print("train retrieval labels:", int(train_batch.item_targets.ne(IGNORE_INDEX).sum()))
assert not valid_batch.action_targets[:, : 2 * TRAIN_CUTOFF].ne(IGNORE_INDEX).any()



train tokens: torch.Size([256, 28])
train action labels: 3584
train retrieval labels: 2842


## 4. HSTU layer с нуля

Для $X\in\mathbb R^{B\times N\times d}$ одна fused-проекция создаёт $U,V,Q,K$. Pointwise attention:

$$A=\operatorname{SiLU}(QK^\top/\sqrt{d_h}+r_{position,time})\odot M_{causal}.$$

В отличие от softmax, сумма весов не обязана быть 1 и сохраняет intensity истории. После $AV$ обязателен LayerNorm; затем результат умножается на gate $U$, проецируется и складывается с residual. Для устойчивости tiny-обучения перед fused-проекцией добавлен обычный pre-LayerNorm; это явно обозначенное инженерное упрощение, которого нет в записи уравнения (1) статьи.



In [4]:
class HSTULayer(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_len, dropout=0.0):
        super().__init__()
        if d_model % num_heads:
            raise ValueError("d_model должен делиться на num_heads")
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.max_seq_len = max_seq_len

        self.input_norm = nn.LayerNorm(d_model)
        self.uvqk = nn.Linear(d_model, 4 * d_model)
        self.position_bias = nn.Embedding(max_seq_len, num_heads)
        self.time_decay_raw = nn.Parameter(torch.zeros(num_heads))
        self.pool_norm = nn.LayerNorm(d_model)
        self.output = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

        nn.init.zeros_(self.position_bias.weight)

    def _split_heads(self, tensor):
        batch, length, _ = tensor.shape
        return tensor.view(batch, length, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, timestamps, padding_mask, position_ids=None, pair_mask=None):
        batch, length, _ = x.shape
        if position_ids is None:
            position_ids = torch.arange(length, device=x.device).expand(batch, -1)

        projected = F.silu(self.uvqk(self.input_norm(x)))
        u, v, q, k = [self._split_heads(part) for part in projected.chunk(4, dim=-1)]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        relative_position = (position_ids[:, :, None] - position_ids[:, None, :]).clamp(
            min=0, max=self.max_seq_len - 1
        )
        positional = self.position_bias(relative_position).permute(0, 3, 1, 2)

        delta_time = (timestamps[:, :, None] - timestamps[:, None, :]).clamp_min(0.0)
        decay = F.softplus(self.time_decay_raw).view(1, self.num_heads, 1, 1)
        temporal = -decay * torch.log1p(delta_time).unsqueeze(1)

        if pair_mask is None:
            causal = torch.ones(length, length, dtype=torch.bool, device=x.device).tril()
            allowed = causal.view(1, 1, length, length)
        else:
            allowed = pair_mask[:, None, :, :]
        allowed = allowed & padding_mask[:, None, None, :] & padding_mask[:, None, :, None]

        weights = F.silu(scores + positional + temporal)
        weights = weights.masked_fill(~allowed, 0.0)
        pooled = torch.matmul(weights, v).transpose(1, 2).reshape(batch, length, self.d_model)
        gated = self.pool_norm(pooled) * u.transpose(1, 2).reshape(batch, length, self.d_model)
        result = x + self.dropout(self.output(gated))
        return result * padding_mask.unsqueeze(-1)


class HSTUEncoder(nn.Module):
    def __init__(self, vocab_size, num_items, d_model=48, num_heads=4, num_layers=2, max_seq_len=96):
        super().__init__()
        self.num_items = num_items
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.type_embedding = nn.Embedding(2, d_model)
        self.layers = nn.ModuleList(
            [HSTULayer(d_model, num_heads, max_seq_len) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.action_head = nn.Linear(d_model, 2)

    def forward(self, batch, position_ids=None, pair_mask=None):
        hidden = self.token_embedding(batch.token_ids) + self.type_embedding(batch.type_ids)
        hidden = hidden * batch.padding_mask.unsqueeze(-1)
        for layer in self.layers:
            hidden = layer(
                hidden,
                batch.timestamps,
                batch.padding_mask,
                position_ids=position_ids,
                pair_mask=pair_mask,
            )
        hidden = self.final_norm(hidden)
        action_logits = self.action_head(hidden)
        # Weight tying: output items используют те же vectors, что input item tokens.
        item_weights = self.token_embedding.weight[ITEM_OFFSET : ITEM_OFFSET + self.num_items]
        item_logits = torch.matmul(hidden, item_weights.t())
        return {"hidden": hidden, "action_logits": action_logits, "item_logits": item_logits}


hstu = HSTUEncoder(VOCAB_SIZE, NUM_ITEMS).to(DEVICE)
mini = PackedBatch(**{name: value[:4] for name, value in train_batch.__dict__.items()}).to(DEVICE)
with torch.no_grad():
    outputs = hstu(mini)
print({name: tuple(value.shape) for name, value in outputs.items()})
assert outputs["hidden"].shape == (4, 2 * TRAIN_CUTOFF, 48)
assert outputs["action_logits"].shape[-1] == 2
assert outputs["item_logits"].shape[-1] == NUM_ITEMS



{'hidden': (4, 28, 48), 'action_logits': (4, 28, 2), 'item_logits': (4, 28, 48)}


## 5. Transformer baseline с теми же входами и heads

Baseline отличается core-блоком: softmax self-attention + FFN. Embedding, target masks и item-weight tying совпадают. Это небольшой sanity baseline, а не точное воспроизведение настроек SASRec из статьи.



In [5]:
class TransformerBaseline(nn.Module):
    def __init__(self, vocab_size, num_items, d_model=48, num_heads=4, num_layers=2, max_seq_len=96):
        super().__init__()
        self.num_items = num_items
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_ID)
        self.type_embedding = nn.Embedding(2, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=2 * d_model,
            dropout=0.0,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers, enable_nested_tensor=False)
        self.final_norm = nn.LayerNorm(d_model)
        self.action_head = nn.Linear(d_model, 2)

    def forward(self, batch, position_ids=None, pair_mask=None):
        if pair_mask is not None:
            raise NotImplementedError("Custom M-FALCON mask проверяется только для HSTU")
        length = batch.token_ids.size(1)
        positions = torch.arange(length, device=batch.token_ids.device)
        hidden = (
            self.token_embedding(batch.token_ids)
            + self.type_embedding(batch.type_ids)
            + self.position_embedding(positions)[None, :, :]
        )
        causal_mask = torch.ones(length, length, dtype=torch.bool, device=hidden.device).triu(1)
        hidden = self.encoder(
            hidden,
            mask=causal_mask,
            src_key_padding_mask=~batch.padding_mask,
            is_causal=True,
        )
        hidden = self.final_norm(hidden)
        item_weights = self.token_embedding.weight[ITEM_OFFSET : ITEM_OFFSET + self.num_items]
        return {
            "hidden": hidden,
            "action_logits": self.action_head(hidden),
            "item_logits": torch.matmul(hidden, item_weights.t()),
        }


transformer = TransformerBaseline(VOCAB_SIZE, NUM_ITEMS).to(DEVICE)
print("HSTU parameters:", sum(p.numel() for p in hstu.parameters()))
print("Transformer parameters:", sum(p.numel() for p in transformer.parameters()))



HSTU parameters: 27418
Transformer parameters: 45266


## 6. Masked multi-task loss и метрики

$$\mathcal L=\mathcal L_{action}+\lambda\mathcal L_{next\ item}.$$

Для ranking считаем Normalized Entropy (ниже лучше). Для retrieval — full-catalog HR@K и NDCG@K: sampled negatives здесь нет.



In [6]:
def compute_loss(outputs, batch, retrieval_weight=0.7):
    action_loss = F.cross_entropy(
        outputs["action_logits"].reshape(-1, 2),
        batch.action_targets.reshape(-1),
        ignore_index=IGNORE_INDEX,
    )
    item_loss = F.cross_entropy(
        outputs["item_logits"].reshape(-1, NUM_ITEMS),
        batch.item_targets.reshape(-1),
        ignore_index=IGNORE_INDEX,
    )
    return action_loss + retrieval_weight * item_loss, action_loss, item_loss


def subset_packed(batch, indices):
    return PackedBatch(**{name: value[indices] for name, value in batch.__dict__.items()})


def retrieval_metrics(logits, targets, ks=(5, 10)):
    valid = targets.ne(IGNORE_INDEX)
    scores = logits[valid]
    truth = targets[valid]
    order = scores.argsort(dim=-1, descending=True)
    ranks = order.eq(truth[:, None]).float().argmax(dim=-1) + 1
    metrics = {}
    for k in ks:
        metrics[f"HR@{k}"] = ranks.le(k).float().mean().item()
        metrics[f"NDCG@{k}"] = (
            ranks.le(k).float() / torch.log2(ranks.float() + 1.0)
        ).mean().item()
    return metrics


@torch.no_grad()
def evaluate(model, batch):
    model.eval()
    data = batch.to(DEVICE)
    outputs = model(data)
    total, action_loss, item_loss = compute_loss(outputs, data)

    action_valid = data.action_targets.ne(IGNORE_INDEX)
    labels = data.action_targets[action_valid]
    prevalence = labels.float().mean().clamp(1e-6, 1 - 1e-6)
    entropy = -(prevalence * prevalence.log() + (1 - prevalence) * (1 - prevalence).log())
    metrics = {
        "loss": total.item(),
        "action_NE": action_loss.item() / entropy.item(),
        "action_accuracy": outputs["action_logits"][action_valid].argmax(-1).eq(labels).float().mean().item(),
        "item_CE": item_loss.item(),
    }
    metrics.update(retrieval_metrics(outputs["item_logits"], data.item_targets))
    return metrics


loss, action_loss, item_loss = compute_loss(hstu(mini), mini)
loss.backward()
assert all(torch.isfinite(p.grad).all() for p in hstu.parameters() if p.grad is not None)
hstu.zero_grad(set_to_none=True)
print(f"finite-gradient smoke test passed; loss={loss.item():.3f}")



finite-gradient smoke test passed; loss=10.776


## 7. Обучение HSTU и baseline

Один epoch видит каждый user-prefix один раз; supervision есть на всех causal target positions. Это имитирует generative training: история не кодируется заново для каждой метки.



In [7]:
def train_model(model, batch, epochs=12, batch_size=32, learning_rate=3e-3):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    history = []
    generator = torch.Generator().manual_seed(SEED)

    for epoch in range(1, epochs + 1):
        model.train()
        permutation = torch.randperm(batch.token_ids.size(0), generator=generator)
        running = 0.0
        for start in range(0, len(permutation), batch_size):
            indices = permutation[start : start + batch_size]
            mini_batch = subset_packed(batch, indices).to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            outputs = model(mini_batch)
            loss, _, _ = compute_loss(outputs, mini_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item() * len(indices)
        epoch_loss = running / len(permutation)
        history.append(epoch_loss)
        if epoch in {1, epochs // 2, epochs}:
            print(f"epoch={epoch:02d} train_loss={epoch_loss:.4f}")
    return history


torch.manual_seed(SEED)
hstu = HSTUEncoder(VOCAB_SIZE, NUM_ITEMS).to(DEVICE)
print("\nHSTU")
hstu_history = train_model(hstu, train_batch)

torch.manual_seed(SEED)
transformer = TransformerBaseline(VOCAB_SIZE, NUM_ITEMS).to(DEVICE)
print("\nTransformer")
transformer_history = train_model(transformer, train_batch)




HSTU
epoch=01 train_loss=8.3376
epoch=06 train_loss=2.6482
epoch=12 train_loss=1.7142

Transformer
epoch=01 train_loss=7.2386
epoch=06 train_loss=3.1203
epoch=12 train_loss=2.4909


In [8]:
hstu_metrics = evaluate(hstu, valid_batch)
transformer_metrics = evaluate(transformer, valid_batch)

print("\nTemporal validation (synthetic; не числа статьи)")
print(f"{'metric':<18} {'HSTU':>10} {'Transformer':>12}")
for metric in hstu_metrics:
    print(f"{metric:<18} {hstu_metrics[metric]:>10.4f} {transformer_metrics[metric]:>12.4f}")




Temporal validation (synthetic; не числа статьи)
metric                   HSTU  Transformer
loss                   1.9843       3.4822
action_NE              1.0166       1.0062
action_accuracy        0.8613       0.8584
item_CE                2.2501       4.3960
HR@5                   0.6777       0.2184
NDCG@5                 0.6364       0.1335
HR@10                  0.7545       0.3630
NDCG@10                0.6608       0.1800


Интерпретация корректна даже если на конкретном seed tiny Transformer оказался лучше по части метрик: PoC проверяет механику, а не доказывает superiority HSTU. Для научного вывода нужны несколько seeds, доверительные интервалы, hyperparameter budget и реальные temporal datasets.



## 8. Stochastic Length: безопасно удаляем далёкие пары

Удаляем **целые item/action пары**, всегда сохраняем последние `recent` событий и случайную часть далёкой истории. Targets выбираются теми же индексами, поэтому alignment не ломается. Это regularizer/демонстрация preprocessing; ускорение custom ragged kernel статьи здесь не воспроизводится.



In [9]:
def stochastic_length(batch, keep_probability=0.45, recent=4, seed=123):
    total_steps = batch.token_ids.size(1) // 2
    generator = torch.Generator().manual_seed(seed)
    keep_event = torch.rand(total_steps, generator=generator).lt(keep_probability)
    keep_event[-recent:] = True
    event_indices = keep_event.nonzero(as_tuple=False).flatten()
    token_indices = torch.stack((2 * event_indices, 2 * event_indices + 1), dim=1).flatten()
    shortened = PackedBatch(**{name: value[:, token_indices] for name, value in batch.__dict__.items()})
    return shortened, event_indices


short_batch, kept = stochastic_length(train_batch)
print("kept event positions:", kept.tolist())
print("token length:", train_batch.token_ids.size(1), "->", short_batch.token_ids.size(1))
with torch.no_grad():
    short_outputs = hstu(subset_packed(short_batch, torch.arange(8)).to(DEVICE))
assert torch.isfinite(short_outputs["hidden"]).all()



kept event positions: [0, 2, 4, 6, 7, 8, 10, 11, 12, 13]
token length: 28 -> 20


## 9. Тест causal mask

Меняем suffix token и проверяем, что representations более раннего prefix не изменились. Это прямой тест отсутствия future leakage.



In [10]:
hstu.eval()
causal_sample = subset_packed(train_batch, torch.tensor([0])).to(DEVICE)
changed = copy.deepcopy(causal_sample)
changed.token_ids = changed.token_ids.clone()
changed.token_ids[:, -1] = ACTION_OFFSET + 1 - all_actions[0, TRAIN_CUTOFF - 1].to(DEVICE)

with torch.no_grad():
    original_hidden = hstu(causal_sample)["hidden"]
    changed_hidden = hstu(changed)["hidden"]
prefix_difference = (original_hidden[:, :-1] - changed_hidden[:, :-1]).abs().max().item()
print("max prefix difference after suffix mutation:", prefix_difference)
assert prefix_difference < 1e-6



max prefix difference after suffix mutation: 0.0


## 10. M-FALCON-inspired candidate isolation

В joint pass все candidates добавлены после общей history, имеют один logical position/time и могут видеть history + самих себя, но **не соседних candidates**. Сравниваем с отдельным forward каждого кандидата. Равенство scores — необходимый correctness invariant M-FALCON. Реальный speedup потребует KV-cache и fused kernel, которых здесь нет.



In [11]:
def append_candidates(base, candidate_items):
    # base содержит одного пользователя без padding.
    num_candidates = len(candidate_items)
    candidate_tokens = torch.tensor(candidate_items, device=DEVICE).view(1, -1) + ITEM_OFFSET
    token_ids = torch.cat([base.token_ids, candidate_tokens], dim=1)
    type_ids = torch.cat(
        [base.type_ids, torch.zeros(1, num_candidates, dtype=torch.long, device=DEVICE)], dim=1
    )
    next_time = base.timestamps[:, -1:] + 1.0
    timestamps = torch.cat([base.timestamps, next_time.expand(1, num_candidates)], dim=1)
    padding_mask = torch.ones_like(token_ids, dtype=torch.bool)
    empty_targets = torch.full_like(token_ids, IGNORE_INDEX)
    batch = PackedBatch(token_ids, type_ids, timestamps, padding_mask, empty_targets, empty_targets.clone())

    history_length = base.token_ids.size(1)
    total_length = token_ids.size(1)
    pair_mask = torch.zeros(1, total_length, total_length, dtype=torch.bool, device=DEVICE)
    pair_mask[:, :history_length, :history_length] = torch.ones(
        history_length, history_length, dtype=torch.bool, device=DEVICE
    ).tril()
    for offset in range(num_candidates):
        row = history_length + offset
        pair_mask[:, row, :history_length] = True
        pair_mask[:, row, row] = True

    position_ids = torch.arange(total_length, device=DEVICE).view(1, -1)
    position_ids[:, history_length:] = history_length  # одинаковый logical target position
    return batch, position_ids, pair_mask


base_length = 12
base = PackedBatch(
    token_ids=causal_sample.token_ids[:, :base_length],
    type_ids=causal_sample.type_ids[:, :base_length],
    timestamps=causal_sample.timestamps[:, :base_length],
    padding_mask=causal_sample.padding_mask[:, :base_length],
    action_targets=causal_sample.action_targets[:, :base_length],
    item_targets=causal_sample.item_targets[:, :base_length],
)
candidates = [2, 11, 25, 39]

with torch.no_grad():
    naive_scores = []
    for candidate in candidates:
        one, positions, mask = append_candidates(base, [candidate])
        probability = hstu(one, position_ids=positions, pair_mask=mask)["action_logits"][:, -1].softmax(-1)[:, 1]
        naive_scores.append(probability.item())

    joint, joint_positions, joint_mask = append_candidates(base, candidates)
    joint_scores = hstu(joint, position_ids=joint_positions, pair_mask=joint_mask)["action_logits"][
        :, -len(candidates) :
    ].softmax(-1)[0, :, 1]

naive_scores = torch.tensor(naive_scores)
print("separate scores:", [round(x, 6) for x in naive_scores.tolist()])
print("joint scores:   ", [round(x, 6) for x in joint_scores.cpu().tolist()])
print("max difference:", (naive_scores - joint_scores.cpu()).abs().max().item())
assert torch.allclose(naive_scores, joint_scores.cpu(), atol=1e-5)



separate scores: [0.859386, 0.887942, 0.8731, 0.86697]
joint scores:    [0.859387, 0.887942, 0.8731, 0.86697]
max difference: 1.1920928955078125e-07


## 11. CPU latency microbenchmark

Это только локальный smoke benchmark при фиксированных $B,N,d$; он не воспроизводит FlashAttention2/H100/ragged результаты статьи. Перед замером делаем warm-up и используем `eval()`.



In [12]:
def benchmark(model, batch, repeats=40):
    model.eval()
    data = subset_packed(batch, torch.arange(32)).to(DEVICE)
    with torch.no_grad():
        for _ in range(5):
            model(data)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        started = time.perf_counter()
        for _ in range(repeats):
            model(data)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - started) / repeats


print(f"HSTU:        {benchmark(hstu, train_batch):.3f} ms / batch")
print(f"Transformer: {benchmark(transformer, train_batch):.3f} ms / batch")



HSTU:        2.399 ms / batch
Transformer: 2.105 ms / batch


## 12. Вывод

Реализованы и проверены ключевые механические утверждения статьи:

- item/action stream даёт target-aware ranking без future leakage;
- один causal pass создаёт supervision на многих позициях;
- HSTU использует pointwise aggregation, time/position bias, post-pooling norm и gate;
- action CE и next-item CE обучаются совместно;
- Stochastic Length сохраняет пары и recent context;
- candidate-isolated joint scoring совпадает с отдельными forward passes.

Следующий честный эксперимент — заменить synthetic data на MovieLens-1M с temporal leave-one-out, добавить несколько seeds и confidence intervals. Для system claims потребуются ragged kernels и GPU/KV-cache benchmark; dense PoC их не доказывает.
